In [1]:
# COLLAPSED
import plotly.graph_objects as go
import torch
from plotly.subplots import make_subplots

from nerfstudio.cameras.rays import RayBundle
from nerfstudio.model_components import ray_samplers

num_samples = 1000
near = 2
far = 5
train_stratified = False

samplers = [
    ray_samplers.UniformSampler,
    ray_samplers.LinearDisparitySampler,
    ray_samplers.SqrtSampler,
    ray_samplers.LogSampler,
]

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=("Uniform", "Linear in Disparity", "Square Root", "Log Sampler"),
    shared_xaxes=True,
    shared_yaxes=True,
    vertical_spacing=0.1,
)

for i, Sampler in enumerate(samplers):
    sampler = Sampler(num_samples=num_samples, train_stratified=train_stratified)

    ray_bundle = RayBundle(
        origins=torch.ones([1, 3]),
        directions=torch.ones([1, 3]),
        pixel_area=torch.ones([1, 1]),
        nears=torch.ones([1, 1]) * near,
        fars=torch.ones([1, 1]) * far,
    )

    samples = sampler.generate_ray_samples(ray_bundle)

    trace = go.Histogram(x=samples.frustums.starts[0, :, 0], nbinsx=50)
    fig.append_trace(trace, i // 2 + 1, i % 2 + 1)

fig.update_yaxes(title_text="# Samples", row=1, col=1)
fig.update_yaxes(title_text="# Samples", row=2, col=1)
fig.update_xaxes(title_text="Distance", row=2, col=1)
fig.update_xaxes(title_text="Distance", row=2, col=2)

# Overlay both histograms
fig.update_layout(height=700, hovermode=False, showlegend=False, margin=dict(l=20, r=20, t=50, b=20))
fig.update_yaxes(range=[0, 80])
fig.update_traces(opacity=0.7)
fig.show()

In [1]:
"""
Default test to make sure train runs
"""

from __future__ import annotations

from pathlib import Path

import pytest

from nerfstudio.configs.method_configs import method_configs
from nerfstudio.data.dataparsers.blender_dataparser import BlenderDataParserConfig
from nerfstudio.data.dataparsers.minimal_dataparser import MinimalDataParserConfig
from nerfstudio.engine.trainer import TrainerConfig
from vanilla_nerf import VanillaModelConfig
from nerfstudio.scripts.train import train_loop

BLACKLIST = [
    "vanilla-nerf"
]


def set_reduced_config(config: TrainerConfig, tmp_path: Path):
    """Reducing the config settings to speedup test"""
    config.machine.device_type = "cuda"
    if hasattr(config.pipeline.model, "implementation"):
        setattr(config.pipeline.model, "implementation", "torch")
    config.mixed_precision = False
    config.use_grad_scaler = False
    config.max_num_iterations = 2
    # reduce dataset factors; set dataset to test
    config.pipeline.datamanager.dataparser = BlenderDataParserConfig(data=Path("/mnt/cc7c68c6-c81c-401d-91fd-04c40177514b/ActiveNeRF/data/nerf_synthetic/lego/"))
    config.pipeline.datamanager.train_num_images_to_sample_from = 1
    config.pipeline.datamanager.train_num_rays_per_batch = 4
    
    # use tensorboard logging instead of wandb
    config.vis = "tensorboard"
    config.logging.relative_log_dir = Path("/tmp/")

    # reduce model factors
    if hasattr(config.pipeline.model, "num_coarse_samples"):
        assert isinstance(config.pipeline.model, VanillaModelConfig)
        config.pipeline.model.num_coarse_samples = 4
    if hasattr(config.pipeline.model, "num_importance_samples"):
        assert isinstance(config.pipeline.model, VanillaModelConfig)
        config.pipeline.model.num_importance_samples = 4
    # remove viewer
    config.viewer.quit_on_train_completion = True

    # timestamp & output directory
    config.set_timestamp()
    config.output_dir = tmp_path / "outputs"

    return config


@pytest.mark.filterwarnings("ignore::DeprecationWarning")
def test_train(tmp_path: Path):
    """test run train script works properly"""
    all_config_names = method_configs.keys()
    for config_name in all_config_names:
        if config_name in BLACKLIST:
            print("skipping", config_name)
            continue
        config = method_configs[config_name]

        # solo VanillaModelConfig
        if not isinstance(config.pipeline.model, VanillaModelConfig):
            print("skipping non-vanilla config", config_name)
            print(f"testing run for: {config_name}")
            config = set_reduced_config(config, tmp_path)
            train_loop(local_rank=0, world_size=0, config=config)
            continue

        


def test_simple_io(tmp_path: Path):
    """test to check minimal data IO works correctly"""
    config = method_configs["vanilla-nerf"]
    config.pipeline.datamanager.dataparser = MinimalDataParserConfig(data=Path("tests/data/minimal_parser"))
    config = set_reduced_config(config, tmp_path)
    train_loop(local_rank=0, world_size=0, config=config)


if __name__ == "__main__":
    tmp_path= Path("temporal")
    test_train( tmp_path=tmp_path)
    #test_simple_io(tmp_path=tmp_path)

ModuleNotFoundError: No module named 'pytest'

In [2]:
from nerfstudio.configs import TrainerConfig
from nerfstudio.scripts.train import set_reduced_config
from pathlib import Path

tmp_path = Path("/tmp/test_run")
config = TrainerConfig.load_from_file("configs/reduced/vanilla_nerf.yaml")
config = set_reduced_config(config, tmp_path)

# entrenar
trainer = config.setup_trainer()
trainer.train()


ImportError: cannot import name 'TrainerConfig' from 'nerfstudio.configs' (/home/sknkllr/miniconda3/envs/o3d/lib/python3.11/site-packages/nerfstudio/configs/__init__.py)

# Vanilla model working

This vanilla model provides 

In [1]:
from pathlib import Path
from vanilla_nerf import VanillaModelConfig
from nerfstudio.configs.method_configs import method_configs
from nerfstudio.scripts.train import train_loop
from nerfstudio.data.dataparsers.blender_dataparser import BlenderDataParserConfig

# Obtener la config de vanilla nerf
config = method_configs["vanilla-nerf"]

# Reemplazar el modelo con una instancia nueva (opcionalmente puedes ajustar parámetros manualmente)
config.pipeline.model = VanillaModelConfig(
    num_coarse_samples=4,       # opcional: ajusta según quieras
    num_importance_samples=4    # opcional
)
config.pipeline.datamanager.dataparser = BlenderDataParserConfig(data=Path("/home/skunkll/data/nerf_synthetic/lego/"))#"/mnt/cc7c68c6-c81c-401d-91fd-04c40177514b/ActiveNeRF/data/nerf_synthetic/lego/"))

# Ejecutar entrenamiento
train_loop(local_rank=0, world_size=1, config=config)


[14:12:32] Saving checkpoints to: outputs/unnamed/vanilla-nerf/{timestamp}/nerfstudio_models              ]8;id=234053;file:///home/skunkll/miniconda3/envs/o3d/lib/python3.11/site-packages/nerfstudio/engine/trainer.py\trainer.py]8;;\:]8;id=146316;file:///home/skunkll/miniconda3/envs/o3d/lib/python3.11/site-packages/nerfstudio/engine/trainer.py#142\142]8;;\

Setting up training dataset...

Caching all 100 images.

Output()

Setting up evaluation dataset...

...

[14:12:49] Printing max of 10 lines. Set flag --logging.local-writer.max-log-size=0 to disable line        ]8;id=67136;file:///home/skunkll/miniconda3/envs/o3d/lib/python3.11/site-packages/nerfstudio/utils/writer.py\writer.py]8;;\:]8;id=354508;file:///home/skunkll/miniconda3/envs/o3d/lib/python3.11/site-packages/nerfstudio/utils/writer.py#449\449]8;;\
           wrapping.                                                                                                    

Step (% Done)       Train Iter (time)    ETA (time)           
--------------------------------------------------------------
Step (% Done)       Train Iter (time)    ETA (time)            
-------------------------------------------------------------- 
0 (0.00%)           289.378 ms           3 d, 8 h, 22 m, 57 s  
Step (% Done)       Train Iter (time)    ETA (time)           Train Rays / Sec      
----------------------------------------------------------------------------------- 
0 (0.00%)           289.378 ms           3 d, 8 h, 22 m, 57 s                       
10 (0.00%)          59.878 ms            16 h, 37 m, 57 s     26.93 K               
Step (% Done)       Train Iter (time)    ETA (time)           Train Rays / Sec      
----------------------------------------------------------------------------------- 
0 (0.00%)           289.378 ms           3 d, 8 h, 22 m, 57 s                       
10 (0.00%)          59.878 ms            16 h, 37 m, 57 s     26.93 K               
20

/home/skunkll/miniconda3/envs/o3d/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:70: FutureWarning: Importing `spectral_angle_mapper` from `torchmetrics.functional` was deprecated and will be removed in 2.0. Import `spectral_angle_mapper` from `torchmetrics.image` instead.
  _future_warning(


Step (% Done)       Train Iter (time)    ETA (time)           Train Rays / Sec     Test Rays / Sec       
-------------------------------------------------------------------------------------------------------- 
410 (0.04%)         33.986 ms            9 h, 26 m, 11 s      30.59 K                                    
420 (0.04%)         34.030 ms            9 h, 26 m, 56 s      30.51 K                                    
430 (0.04%)         33.607 ms            9 h, 19 m, 52 s      31.02 K                                    
440 (0.04%)         33.361 ms            9 h, 15 m, 46 s      31.24 K                                    
450 (0.04%)         33.700 ms            9 h, 21 m, 24 s      30.93 K                                    
460 (0.05%)         34.294 ms            9 h, 31 m, 18 s      30.45 K                                    
470 (0.05%)         36.192 ms            10 h, 2 m, 55 s      29.04 K                                    
480 (0.05%)         37.149 ms            10 h,

KeyboardInterrupt: 

In [1]:
from pathlib import Path
from activenerf_config  import ActiveNeRFModelConfig
from activeconfigs import method_configs
from nerfstudio.scripts.train import train_loop
from activedatamanager import ActiveNeRFDataManager
from nerfstudio.data.dataparsers.blender_dataparser import BlenderDataParserConfig
from nerfstudio.data.datamanagers.base_datamanager import VanillaDataManagerConfig



config = method_configs["activenerf"]
config.pipeline.model = ActiveNeRFModelConfig(
    num_coarse_samples=64,  
    num_importance_samples=128, 
    use_uncertainty=True,  
)
config.pipeline.datamanager = VanillaDataManagerConfig(
    _target=ActiveNeRFDataManager, 
    dataparser=BlenderDataParserConfig(data=Path("/home/skunkll/data/nerf_synthetic/lego/")),
)

train_loop(local_rank=0, world_size=1, config=config)    

[19:11:18] Saving checkpoints to: outputs/unnamed/activenerf/{timestamp}/nerfstudio_models                ]8;id=234053;file:///home/skunkll/miniconda3/envs/o3d/lib/python3.11/site-packages/nerfstudio/engine/trainer.py\trainer.py]8;;\:]8;id=146316;file:///home/skunkll/miniconda3/envs/o3d/lib/python3.11/site-packages/nerfstudio/engine/trainer.py#142\142]8;;\

Setting up training dataset...

Caching all 100 images.

Output()

Setting up evaluation dataset...

...

wandb: Currently logged in as: zaulo (zaulo-cic-ipn) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


logging events to: outputs/unnamed/activenerf/{timestamp}

[19:11:42] Printing max of 10 lines. Set flag --logging.local-writer.max-log-size=0 to disable line        ]8;id=67136;file:///home/skunkll/miniconda3/envs/o3d/lib/python3.11/site-packages/nerfstudio/utils/writer.py\writer.py]8;;\:]8;id=354508;file:///home/skunkll/miniconda3/envs/o3d/lib/python3.11/site-packages/nerfstudio/utils/writer.py#449\449]8;;\
           wrapping.                                                                                                    

Step (% Done)       Train Iter (time)    ETA (time)           
--------------------------------------------------------------
Step (% Done)       Train Iter (time)    ETA (time)             
--------------------------------------------------------------  
0 (0.00%)           1 s, 705.274 ms      3 d, 22 h, 44 m, 14 s  
Step (% Done)       Train Iter (time)    ETA (time)           Train Rays / Sec       
-----------------------------------------------------------------------------------  
0 (0.00%)           1 s, 705.274 ms      3 d, 22 h, 44 m, 14 s                       
10 (0.01%)          708.439 ms           1 d, 15 h, 21 m, 20 s 1.70 K                
Step (% Done)       Train Iter (time)    ETA (time)           Train Rays / Sec       
-----------------------------------------------------------------------------------  
0 (0.00%)           1 s, 705.274 ms      3 d, 22 h, 44 m, 14 s                       
10 (0.01%)          708.439 ms           1 d, 15 h, 21 m, 20 s 1.70 K        

: 